# Lab 3 — Acquiring data from a Web API

**Course:** Data Acquisition, Processing and Mining for AI — week 3

**Objectives.** Write a client that **retries with backoff** instead of
dying, **paginate** a real API to completion, and **store** what you got with
a log of how you got it. Others: sending a request and reading
the result, a disk cache, nested JSON to a table, and authenticating to GitHub with your personal token.

**Your GitHub token.** Exercise 6 uses the GitHub personal access token.
It reads `GITHUB_TOKEN` from the environment, and asks you for it with
`getpass` if that is not set.

**Dataset.** [Open Library](https://openlibrary.org/developers/api) — no key,
1 request/s (3/s if your `User-Agent` identifies you). The query is books on
*machine learning* first published between 2018 and 2024 and should return 478 works.

**If the network is down.** Set `OFFLINE=1` in the environment
before starting Jupyter and every request in this notebook is replayed from
`fixtures/`.

**Core exercises.** Three exercises are **core**, and they are
what the session is for: **2** (network-aware client), **4**
(pagination loop) and **7** (stored dataset + log). These are required for M1.

**How to work.** Each core and extension cell opens with a **skeleton**: the
names and the structure are given, and `...` marks what you should input.

**Project.** This is what will be the main spine of your M1 in the project.

In [ ]:
import getpass
import hashlib
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from requests.structures import CaseInsensitiveDict

SEARCH = "https://openlibrary.org/search.json"
QUERY = 'subject:"machine learning" AND first_publish_year:[2010 TO 2018]'
FIELDS = ("key,title,author_name,first_publish_year,subject,"
          "edition_count,number_of_pages_median")

# Identify yourself
UA = "datacq-lab3 (firstname.lastname@etu.univ-grenoble-alpes.fr)"
HEADERS = {"User-Agent": UA}

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)
FIXTURES = Path("fixtures")

### The offline switch

`OFFLINE=1` replaces
`requests.get` with a lookup in `fixtures/`.

If Open Library starts throttling: run `os.environ["OFFLINE"] = "1"` and re-run the cell below.

In [ ]:
OFFLINE = os.environ.get("OFFLINE") == "1"
POLITE_DELAY = 0.0 if OFFLINE else 1.0   # seconds between live requests


def cache_key(url, params=None):
    """A short, stable name for one (url, params) pair.
    """
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


class FixtureResponse:

    def __init__(self, record):
        self.url = record.get("final_url", record["url"])
        self.status_code = record["status_code"]
        self.headers = CaseInsensitiveDict(record["headers"])
        self._body = record["body"]

    def json(self):
        return self._body

    @property
    def links(self):
        header = self.headers.get("link")
        if not header:
            return {}
        return {l["rel"]: l for l in requests.utils.parse_header_links(header)}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code} for {self.url}")


def offline_get(url, params=None, **kwargs):
    path = FIXTURES / f"{cache_key(url, params)}.json"
    if not path.exists():
        raise RuntimeError(
            f"no fixture for {url} {params}")
    return FixtureResponse(json.loads(path.read_text()))

if OFFLINE:
    requests.get = offline_get
    print("OFFLINE:", len(list(FIXTURES.glob("*.json"))), "fixtures loaded")
else:
    print("live requests")

live requests


## Exercise 1 (solved) — one request

Ask Open Library for the **first 3** works matching `QUERY`, requesting only
the fields in `FIELDS`. Then print, in this order:

1. the status code,
2. the `content-type` header,
3. the **URL that was actually sent** (`r.url`) — look at what happened to the
   quotes and spaces in `QUERY`,
4. `numFound` and the number of documents you got back.

Pass `params=`, `headers=HEADERS` and `timeout=10`.

In [ ]:
r = requests.get(SEARCH,
                 params={"q": QUERY, "limit": 3, "fields": FIELDS},
                 headers=HEADERS, timeout=10)
r.raise_for_status()

payload = r.json()
print("status      :", r.status_code)
print("content-type:", r.headers["content-type"])
print("url sent    :", r.url)
print("numFound    :", payload["numFound"])
print("docs        :", len(payload["docs"]))
payload["docs"][0]

status      : 200
content-type: application/json
url sent    : https://openlibrary.org/search.json?q=subject%3A%22machine+learning%22+AND+first_publish_year%3A%5B2010+TO+2018%5D&limit=3&fields=key%2Ctitle%2Cauthor_name%2Cfirst_publish_year%2Csubject%2Cedition_count%2Cnumber_of_pages_median
numFound    : 760
docs        : 3


{'author_name': ['Ian Goodfellow',
  'Yoshua Bengio',
  'Aaron Courville',
  'Francis Bach'],
 'edition_count': 5,
 'first_publish_year': 2016,
 'key': '/works/OL17801809W',
 'number_of_pages_median': 800,
 'title': 'Deep Learning',
 'subject': ['Machine learning',
  'Apprentissage automatique',
  'Computers and IT',
  'Maschinelles Lernen',
  'Deep learning (Machine learning)',
  'Electronic books',
  'COMPUTERS / Artificial Intelligence / General',
  'Kunstmatige intelligentie',
  'Computer science']}

**Question (\*):** `numFound` and `len(docs)` disagree. What is each one counting, and which of the two belongs in your dataset information?

*Your answer here* docs belongs to my dataset, len(docs) is the total amount of rows available in server


## Exercise 2 — network-aware client

Write `http_get(url, params=None, headers=HEADERS, attempts=5)` returning the
parsed JSON body, with this behaviour:

- `200` → return `r.json()`.
- `429`, `500`, `502`, `503`, `504` → wait, then retry. Wait `Retry-After`
  seconds if the header is there **and positive**; otherwise a random time in
  $[0, 2^k]$, where `k` is the retry number (0, 1, 2, …).
- a **timeout or a dropped connection** (`requests.RequestException`): same
  treatment.
- any other status: `raise_for_status()`, immediately.
- out of attempts: raise a `RuntimeError` naming the URL.

Read `Retry-After` with `int()`. RFC 9110 allows **either** a number of
seconds **or** an HTTP date.

Always pass `timeout=10`. The skeleton gives you the `try`/`except`/`else` shape.

In [ ]:
def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):
        try:
            r = requests.get(url, params=params, headers=headers,
                             timeout=10)
        except requests.RequestException as exc:   # timeout, DNS, reset
            status, retry_after = repr(exc), 0
        else:
            # 1. 200 -> return the parsed body
            if (r.status_code == 200):
                return r.json()

            # 2. a status that will not change -> raise_for_status()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()


            # 3. otherwise -> remember the status, and read Retry-After
            #    (numeric form only; 0 if the header is absent)
            status = r.status_code
            try:
                retry_after = int(r.headers.get("Retry-After", 0))
            except (ValueError, TypeError):
                retry_after = 0


        if k + 1 < attempts:

            if retry_after > 0:
                wait = retry_after
            else:
                wait = random.uniform(0, 2**k)


            print(f"  {status} on attempt {k + 1}, sleeping {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {attempts} attempts: {url}")


Test it against a server that fails twice and then answers.

The cell below is given, and it **fails until your `http_get` works**.

In [ ]:
attempts_made = []


def flaky_get(url, params=None, **kwargs):
    attempts_made.append(url)
    if len(attempts_made) < 3:
        return FixtureResponse({"url": url, "status_code": 503,
                                "headers": {}, "body": {}})
    return FixtureResponse({"url": url, "status_code": 200,
                            "headers": {}, "body": {"ok": True}})


saved_get, requests.get = requests.get, flaky_get
try:
    print("result:", http_get("https://example.invalid/thing"))
finally:
    requests.get = saved_get

print("attempts:", len(attempts_made))
assert len(attempts_made) == 3

  503 on attempt 1, sleeping 0.8s
  503 on attempt 2, sleeping 0.2s
result: {'ok': True}
attempts: 3


**Question:** why is `404` in the "give up immediately" branch while `503` is
in the "retry" branch? Name one status you would move if you were acquiring
from GitHub.I would remove 404 becuase it is code for 'maybe something is here but you cant access it'

*Your answer here*


## Exercise 3 (solved) — caching

The cell below wraps `http_get` in `cached_get(url, params=None)`,
which:

1. computes `cache_key(url, params)`,
2. returns the parsed contents of `cache/<key>.json` if that file exists,
3. otherwise fetches, writes the body to that file, and returns it,
4. waits `POLITE_DELAY` seconds **only when it actually made a request**.

It is then called twice on the same query, to show that the second call cost
zero requests.

In [ ]:
requests_made = 0


def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"
    if path.exists():
        return json.loads(path.read_text())

    global requests_made
    requests_made += 1
    body = http_get(url, params)
    path.write_text(json.dumps(body))
    time.sleep(POLITE_DELAY)
    return body


probe = {"q": QUERY, "limit": 3, "fields": FIELDS}
first = cached_get(SEARCH, probe)
second = cached_get(SEARCH, probe)

print("identical    :", first == second)
print("requests made:", requests_made)
print("cache files  :", len(list(CACHE.glob("*.json"))))

identical    : True
requests made: 1
cache files  : 1


**Question (\*):** the cache key is built from the URL **and** the parameters. What
breaks if you key on the URL alone? And what breaks if you key on
`str(params)` instead of `json.dumps(params, sort_keys=True)`?

*Your answer here*


## Exercise 4 — pagination loop

Open Library pages with `limit` (page size, max 100) and `page` (1-based).

Write a loop that collects **every** work matching `QUERY`, fetching with
`cached_get`:

- 100 per page, starting at page 1,
- stop when you have `numFound` works **or** the server returns an empty page,
- stop at `MAX_PAGES = 10` regardless,
- print the page number and the running total as you go.

Report `numFound`, how many you collected, and how many pages it took and whether the two agree.

In [ ]:
MAX_PAGES = 10
PAGE_LIMIT = 100


works, page, num_found = [], 1, None

pages_fetched = 0

# loop: fetch a page, extend works, stop at numFound, an empty page,
# or MAX_PAGES

for page in range(1, MAX_PAGES):
    params = {
        "q": QUERY,
        "fields": FIELDS,
        "limit": PAGE_LIMIT,
        "page": page,
    }

    data = cached_get(SEARCH, params=params)

    # Capture total available results on the first request
    if num_found is None:
        num_found = data.get("numFound", 0)

    docs = data.get("docs", [])

    # Stop condition: server returned an empty page
    if not docs:
        break

    works.extend(docs)
    pages_fetched = page
    print(f"Page {page}: collected {len(docs)} items (running total: {len(works)})")

    # Stop condition: collected all available matching works
    if len(works) >= num_found:
        break

# then report numFound, how many you collected, how many pages it
# took, and whether the first two agree
collected = len(works)
agreed = (collected == num_found)

print("\n--- Summary ---")
print(f"numFound:        {num_found}")
print(f"Works collected: {collected}")
print(f"Pages fetched:   {pages_fetched}")
print(f"Counts agree:    {agreed}")


Page 1: collected 100 items (running total: 100)
Page 2: collected 100 items (running total: 200)
Page 3: collected 100 items (running total: 300)
Page 4: collected 100 items (running total: 400)
Page 5: collected 100 items (running total: 500)
Page 6: collected 100 items (running total: 600)
Page 7: collected 100 items (running total: 700)
Page 8: collected 60 items (running total: 760)

--- Summary ---
numFound:        760
Works collected: 760
Pages fetched:   8
Counts agree:    True


**Question:** suppose a librarian adds a book to this subject while your loop
is between page 2 and page 3. What happens to your dataset, and which of the
two pagination styles from the lecture would have prevented it?

Counts no longer agree: coursor would have prevented it.

## Exercise 5 (solved) — from JSON to table

The two cells below turn `works` into the two tables:

1. a DataFrame built from `works` with `pd.json_normalize`;
2. `subject` is a **list per work**, resulting in a second long table with one row per
   `(work, subject)` pair, the 10 most frequent subjects, and the distinct
   subject count before and after lowercasing.

In [ ]:
df = pd.json_normalize(works)
print("shape:", df.shape)
print()
print(df.dtypes)
print()
print("missing per column:")
print(df.isna().sum())
df.head(3)

shape: (760, 7)

author_name                object
edition_count               int64
first_publish_year          int64
key                        object
number_of_pages_median    float64
title                      object
subject                    object
dtype: object

missing per column:
author_name                 1
edition_count               0
first_publish_year          0
key                         0
number_of_pages_median    128
title                       0
subject                     0
dtype: int64


,author_name,edition_count,first_publish_year,key,number_of_pages_median,title,subject
0,"[Ian Goodfellow, Yoshua Bengio, Aaron Courvill...",5,2016,/works/OL17801809W,800.0,Deep Learning,"[Machine learning, Apprentissage automatique, ..."
1,[Ethem Alpaydin],4,2016,/works/OL19723604W,224.0,Machine learning,"[Artificial intelligence, Machine learning, No..."
2,"[John Paul Mueller, Luca Massaron]",6,2016,/works/OL19546858W,432.0,Machine Learning For Dummies,"[Machine learning, Science]"


In [ ]:
subjects = (df[["key", "title", "first_publish_year", "subject"]]
            .explode("subject")
            .dropna(subset=["subject"]))

print("work rows   :", len(df))
print("subject rows:", len(subjects))
print("subjects per work:", round(len(subjects) / len(df), 2))
print()
print(subjects["subject"].value_counts().head(10))
print()
print("distinct subjects            :", subjects["subject"].nunique())
print("distinct, lowercased         :",
      subjects["subject"].str.lower().nunique())

subjects.head(10)

work rows   : 760
subject rows: 3796
subjects per work: 4.99

subject
Machine learning                            755
Artificial intelligence                     190
Data mining                                 148
Computer science                             86
Python (computer program language)           61
Artificial Intelligence (incl. Robotics)     58
Apprentissage automatique                    52
COMPUTERS                                    51
Neural networks (computer science)           44
General                                      44
Name: count, dtype: int64

distinct subjects            : 790
distinct, lowercased         : 740


,key,title,first_publish_year,subject
0,/works/OL17801809W,Deep Learning,2016,Machine learning
0,/works/OL17801809W,Deep Learning,2016,Apprentissage automatique
0,/works/OL17801809W,Deep Learning,2016,Computers and IT
0,/works/OL17801809W,Deep Learning,2016,Maschinelles Lernen
0,/works/OL17801809W,Deep Learning,2016,Deep learning (Machine learning)
0,/works/OL17801809W,Deep Learning,2016,Electronic books
0,/works/OL17801809W,Deep Learning,2016,COMPUTERS / Artificial Intelligence / General
0,/works/OL17801809W,Deep Learning,2016,Kunstmatige intelligentie
0,/works/OL17801809W,Deep Learning,2016,Computer science
1,/works/OL19723604W,Machine learning,2016,Artificial intelligence


**Question (\*):** look at the top subjects. Two problems are visible before any
cleaning.

5 books DON'T have machine learning as a subject, subject names are ambiguous, we could be missing books

**Question (\*):** you now have two tables. Which one is "the dataset"?

*Your answer here*
subjects has been filtered, it contains the data of interest

## Exercise 6 (\*) — GitHub with PAT
The cell below reads `GITHUB_TOKEN` from the environment and falls back to
`getpass.getpass()` — the same two lines as the lecture. Either way the token
is never written into a cell, and never printed.

Send `X-GitHub-Api-Version` on every call. Without it GitHub applies whatever
its current default is; with it, you know which API answered you and can
write that down.

**6.1** Call `https://api.github.com/rate_limit` and print
`x-ratelimit-limit`, `-remaining` and `-reset` (convert the reset to a
readable time). Run it once without the `Authorization` header and once with
it, and compare the limits.

**6.2** Fetch the first **two** pages of issues of `duckdb/duckdb` with
`per_page=5`, following the `Link` header rather than incrementing a page
number. Print each item's number and title — and count how many of them carry
a `pull_request` key.

In [ ]:
TOKEN = os.environ.get("GITHUB_TOKEN")
if not TOKEN and not OFFLINE:
    TOKEN = getpass.getpass("GitHub PAT (Enter to skip): ") or None
GH = {"User-Agent": UA,
      "Accept": "application/vnd.github+json",
      "X-GitHub-Api-Version": "2022-11-28"}


def show_limits(label, headers):
    url = "https://api.github.com/rate_limit"
    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()

    limit = r.headers.get("x-ratelimit-limit")
    remaining = r.headers.get("x-ratelimit-remaining")
    reset_epoch = int(r.headers.get("x-ratelimit-reset", 0))

    print(f"[{label}]")
    print(f"  Limit:     {limit}")
    print(f"  Remaining: {remaining}")



show_limits("no token", GH)
if TOKEN:
    auth_headers = {**GH, "Authorization": f"Bearer {TOKEN}"}
    show_limits("with token", auth_headers)
else:
    print("with token       — GITHUB_TOKEN not set, skipped")


GitHub PAT (Enter to skip): ········
[no token]
  Limit:     60
  Remaining: 60
[with token]
  Limit:     5000
  Remaining: 5000


In [ ]:
gh_headers = {**GH, "Authorization": f"Bearer {TOKEN}"} if TOKEN else GH

url, params, issues = ("https://api.github.com/repos/duckdb/duckdb/issues",
                       {"state": "all", "per_page": 5}, [])
for page in range(2):                      # cap it while developing
    r = requests.get(url, params=params, headers=gh_headers, timeout=10)
    r.raise_for_status()

    issues.extend(r.json())

    # Advance using the parsed Link header
    if "next" in r.links:
        url = r.links["next"]["url"]
        params = None  # The 'next' URL already contains all query params
    else:
        break

for item in issues:
    kind = "PR " if "pull_request" in item else "   "
    print(f"{kind}#{item['number']:>6} {item['title'][:56]}")


PR # 26133 Fix remote pushdown bump quack
   # 26132 ClientContext::Destroy() can std::terminate() the proces
PR # 26131 Fix division by zero in CSV reader when max_line_size ov
PR # 26130 Fix 64-byte VARIANT strings written as empty short strin
PR # 26129 Fix double quoted in error messages
PR # 26128 `SOME` can be used as substitute for `ANY`
PR # 26127 Uncached parallel reads and compressed regression fix
PR # 26126 bump httpfs
PR # 26125 Map legacy table filter keys in deserialization
PR # 26124 Fix cleanup of unfinalized COPY files


**Question:** you asked for issues and some of what came back are pull
requests. How would you only keep the issues?

*Your answer here* Remove {kind} that is PR


## Exercise 7 — storing the dataset and the log

Save both tables as Parquet, and write an **acquisition log** next to them:
a small JSON file recording what was asked, when, and what were the results.

If your project's API is **versioned**, that version has to be in the log.
Open Library is unversioned, so there is nothing to do in the acquisition for the lab.

`author_name` holds **lists**, and 3 works have none. Check that the round-trip through Parquet preserves both, the list
*values* and the *missing* ones, and look at what Python type comes back.

In [ ]:
df.to_parquet(df)
subjects.to_parquet(subjects)

back = ...
print("shape round-trips   :", ...)
print("list values survive :", ...)
print("missing before/after:", ...)

log = {
    # what was asked, when, and what was collected
}
Path("acquisition_log.json").write_text(json.dumps(log, indent=2))
print(json.dumps(log, indent=2))


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

**Question:** one way would be
`df["author_name"].apply(lambda v: v if isinstance(v, list) else [])` making
the column uniformly a list. What are the problems?

*Your answer here*


**Question:** which single field in that log would you need the most if the
acquisition had to be repeated in six months, and why?

*Your answer here*


## Extension (\*) — OAuth2 client credentials (Twitch)

Only if you registered a Twitch application (it needs 2FA on your account).
Set `TWITCH_CLIENT_ID` and `TWITCH_CLIENT_SECRET` in your environment first —
the secret can be considered a **password**.

The exchange: `POST` the two credentials to the token endpoint, get back a
short-lived `access_token`, then send that token on
every Helix request, alongside a `Client-Id` header.

In [ ]:
cid = os.environ.get("TWITCH_CLIENT_ID")
secret = os.environ.get("TWITCH_CLIENT_SECRET")

if cid and secret and not OFFLINE:
    ...
else:
    print("no Twitch credentials in the environment — skipped")


## On your project data

**M1 is due before the W5 lab**: your `acquisition_log.json`, the
acquisition script, and the shape of each Parquet file. Everything you need is in this notebook.

This week you have to:

1. Read your source's docs for the four things that decide your code:
   **auth**, **pagination style**, **rate limit**, and the **shape of a
   response**.
2. Write the acquisition script as three functions:
   `http_get` (retries), `cached_get` (disk cache), and one paginating loop.
   Copy them from this notebook.
3. Decide **one row per what**, and write it down before you write the
   DataFrame.
4. **Run it at home.**
5. Keep the acquisition log.